## Imports

In [1]:
import pandas as pd
import ast
from google.colab import userdata, drive
from tqdm import tqdm
import typing_extensions as typing
import google.generativeai as genai
import json
import time
import os
import csv
from datetime import datetime, timedelta
from abc import ABC, abstractmethod
from openai import OpenAI


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Extract Data

In [3]:
# 1. Load the data and convert strings back to real Python lists
clinical_trials_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/processed_studies.csv")
# clinical_trials_df['inclusion_criteria'] = clinical_trials_df['inclusion_criteria'].apply(ast.literal_eval)

In [4]:
mimic_patients_df = pd.read_parquet("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/df_diag_final_elegibility.parquet")
print(mimic_patients_df.columns.tolist())

['subject_id', 'hadm_id', 'HPI', 'physical_exam', 'chief_complaint', 'icd_diag_principal', 'name_diag_principal', 'list_procedures', '% Hemoglobin A1c', '25-OH Vitamin D', 'ADP', 'ANCA Titer', 'ARCH-1', 'Absolute Basophil Count', 'Absolute CD3 Count', 'Absolute CD4 Count', 'Absolute CD8 Count', 'Absolute Eosinophil Count', 'Absolute Lymphocyte Count', 'Absolute Monocyte Count', 'Absolute Neutrophil Count', 'Acanthocytes', 'Acetaminophen', 'Acetone', 'Alanine Aminotransferase (ALT)', 'Albumin', 'Alkaline Phosphatase', 'Alpha-Fetoprotein', 'Alveolar-arterial Gradient', 'Ammonia', 'Amylase', 'Anion Gap', 'Anisocytosis', 'Anti-Mitochondrial Antibody', 'Anti-Neutrophil Cytoplasmic Antibody', 'Anti-Nuclear Antibody', 'Anti-Nuclear Antibody, Titer', 'Anti-Smooth Muscle Antibody', 'Anti-Thyroglobulin Antibodies', 'Anticardiolipin Antibody IgG', 'Anticardiolipin Antibody IgM', 'Antithrombin', 'Arachadonic Acid', 'Asparate Aminotransferase (AST)', 'Assist/Control', 'Atypical Lymphocytes', 'Bands

In [5]:
matching_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_report.csv")

In [6]:
len(mimic_patients_df)

2813

## Transform

In [7]:
# @title GROQ JUDGE
class BaseJudge(ABC):

    @abstractmethod
    def generate(self, system_prompt, user_prompt):
        pass

class GroqJudge(BaseJudge):

    def __init__(self, api_key, model="deepseek-r1-distill-llama-70b"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1"
        )
        self.model = model

    def generate(self, system_prompt, user_prompt):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content

In [8]:
# @title Clinical Match Engine

class BatchClinicalMatch:
    def __init__(self, model):
        # genai.configure(api_key=userdata.get(api_key))
        # self.model = genai.GenerativeModel(version_genai)
        self.model = model

    def _process_batch(self, batch_df, inclusion, exclusion):
        # Transformamos o lote de pacientes em uma lista compacta de JSON
        # patients_list = batch_df.to_json(orient='records')

        # 1. Converte o lote para uma lista de dicionários
        raw_patients = batch_df.to_dict(orient='records')

        # 2. LIMPEZA DINÂMICA: Remove campos nulos/vazios de cada paciente
        compact_patients = []
        for p in raw_patients:
            # Mantém apenas o que não é nulo, não é NaN e não é string vazia
            clean_p = {k: v for k, v in p.items() if pd.notna(v) and str(v).strip() != "" and str(v).lower() != "nan"}
            compact_patients.append(clean_p)

        # 3. Transforma a lista limpa em string JSON
        patients_json_string = json.dumps(compact_patients)

        system_prompt = """
          You are a Senior Clinical Trial Coordinator and Data Auditor.
          Your task is to verify which patients are eligible to a clinical trial based on the eligibility criteria.

          Evaluate if the patient truly satisfies the criteria provided for the study.
          Identify if there are any logical gaps, missing data in patient profile, or incorrect inclusion/exclusion decisions.

          Respond ONLY with valid JSON.
            """

        prompt = f"""
        As a clinical trial coordinator, analyze this batch of patient records against the criteria.

        STUDY CRITERIA:
        Inclusion: {inclusion}
        Exclusion: {exclusion}

        PATIENTS LIST (JSON):
        {patients_json_string}

        For each patient, decide if they are ELIGIBLE.
        Consider synonyms (e.g., 'Synthetic Substitute' = 'Prosthesis').
        'No pregnant women' = Males OK, non-pregnant Females OK.

        RETURN ONLY A JSON LIST of eligible patients in this format:
        [
          {{"subject_id": 123, "reason": "Short reason"}},
          {{"subject_id": 456, "reason": "Short reason"}}
        ]
        If NO patients are eligible, return an empty list [].
        """
        try:
            response = self.model.generate(system_prompt, prompt)
            response = response.replace("```json", "").replace("```", "").strip()
            if "<think>" in response:
                response = response.split("</think>")[-1].strip()
            return json.loads(response)
        except Exception as e:
            print(f"Erro na API Groq: {e}")
            return []

    def run_inference(self, studies_df, patients_df, batch_size=50, report_file="matching_report.csv", log_file="audit_log.csv"):
        # --- LÓGICA DE CHECKPOINT ---
        processed_ncts = set()
        if os.path.exists(report_file):
            existing_report = pd.read_csv(report_file)
            processed_ncts = set(existing_report['nct_id'].unique())
            print(f"Retomando progresso: {len(processed_ncts)} estudos já processados.")

        for _, study in tqdm(studies_df.iterrows(), total=len(studies_df), desc="Estudos"):
            nct_id = study['nct_id']

            # Pula se já estiver no arquivo
            if nct_id in processed_ncts:
                continue

            study_eligible_ids = []
            audit_data = []

            for i in range(0, len(patients_df), batch_size):
                print(f" -> Processando lote {i//batch_size + 1} de {len(patients_df)//batch_size + 1}...", end="\r")
                batch = patients_df.iloc[i : i + batch_size]
                results = self._process_batch(batch, study['inclusion_criteria'], study['exclusion_criteria'])

                if results:
                    for res in results:
                        sid = res.get('subject_id')
                        study_eligible_ids.append(sid)
                        audit_data.append({
                            "nct_id": nct_id,
                            "subject_id": sid,
                            "reason": res.get('reason')
                        })

                time.sleep(4) # Respeitando a cota da API

            # --- SALVAMENTO INCREMENTAL ---
            # Salva o relatório principal
            report_row = pd.DataFrame([{
                "nct_id": nct_id,
                "eligible_count": len(study_eligible_ids),
                "subject_ids": json.dumps(study_eligible_ids)
            }])
            report_row.to_csv(report_file, mode='a', header=not os.path.exists(report_file), index=False)

            # Salva o log de auditoria
            if audit_data:
                pd.DataFrame(audit_data).to_csv(log_file, mode='a', header=not os.path.exists(log_file), index=False)

            # Adiciona aos processados para controle em tempo de execução
            processed_ncts.add(nct_id)

        return pd.read_csv(report_file), pd.read_csv(log_file)

In [9]:
## Main

In [10]:
# @title Define Engine
judge = GroqJudge(
        api_key=userdata.get("JudgeOpenRouter"),
        model="openai/gpt-oss-120b"
    )
engine = BatchClinicalMatch(
    # api_key="ParsingGeminiAPI",
    # version_genai = 'gemini-2.5-flash'
    model=judge
)

In [11]:
# @title Subsets data
all_cols = mimic_patients_df.columns.tolist()

cols_obrigatorias = ['subject_id', 'gender', 'age', 'name_diag_principal', 'list_procedures', 'HPI', 'chief_complaint']
to_ignore = ['hadm_id']
cols_com_dados = mimic_patients_df.columns[mimic_patients_df.notna().sum() > 0].tolist()
filtered_cols = [c for c in cols_com_dados if c not in to_ignore and not c.endswith('_FLAG')]
cols_finais = list(set(cols_obrigatorias + filtered_cols))
patients_subset = mimic_patients_df[cols_finais].copy()

def trim_clinical_text(text, limit=600):
    text = str(text)
    if len(text) <= limit:
        return text

    # Pega os primeiros 300 e os últimos 300 caracteres
    half = limit // 2
    return text[:half] + "\n[...] [TEXT CUT] [...]\n" + text[-half:]

# Aplica no seu DataFrame
patients_subset['HPI'] = patients_subset['HPI'].apply(trim_clinical_text)

In [24]:
nct_id_list = matching_df['nct_id'].unique().tolist()
len(nct_id_list)

10

,Factor VII,"Osmolality, Measured",Uric Acid,PT,Lipase,Carboxyhemoglobin,"Reticulocyte Count, Manual",pH,Gentamicin,Phenytoin,...,Triiodothyronine (T3),Quantitative G6PD,Cancer Antigen 27.29,Cortisol,Vitamin B12,"Calcium, Total","Cholesterol, HDL",CD5 %,MCV,icd_diag_principal
0,NaN,NaN,NaN,11.9,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,9.4,NaN,NaN,89.0,I5022
1,NaN,NaN,NaN,11.1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,9.4,43.0,NaN,84.0,I472
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8.9,NaN,NaN,90.0,I2510
3,NaN,NaN,NaN,12.9,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8.6,NaN,NaN,91.0,I2109
4,NaN,NaN,NaN,14.8,NaN,NaN,NaN,7.40,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8.1,NaN,NaN,96.0,I2510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,NaN,NaN,NaN,17.1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8.4,NaN,NaN,84.0,I2119
484,NaN,NaN,NaN,10.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8.5,52.0,NaN,NaN,I2510
485,NaN,NaN,NaN,28.9,NaN,NaN,NaN,7.42,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8.9,NaN,NaN,93.0,I5033
486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,8.8,36.0,NaN,83.0,I214


In [25]:
# @title Run report
for nct_id in nct_id_list[1:]:
    df_patients_subset_filtered = matching_df.copy()
    df_patients_subset_filtered['subject_ids'] = df_patients_subset_filtered['subject_ids'].apply(ast.literal_eval)
    df_patients_subset_filtered = df_patients_subset_filtered[df_patients_subset_filtered['nct_id'] == nct_id].explode('subject_ids')
    df_patients_subset_filtered = df_patients_subset_filtered[['subject_ids']]
    df_patients_subset_filtered = df_patients_subset_filtered.rename(columns={'subject_ids': 'subject_id'})
    df_patients_subset_filtered = df_patients_subset_filtered.drop_duplicates()
    df_patients_subset_filtered = patients_subset.merge(df_patients_subset_filtered, on='subject_id', how='inner')

    print(f"Processando estudo: {nct_id}")
    study_df_to_process = clinical_trials_df[clinical_trials_df['nct_id'] == nct_id]
    report_df, audit_df = engine.run_inference(
        study_df_to_process,
        df_patients_subset_filtered,
        batch_size = 50,
        report_file="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_judge.csv",
        log_file="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/audit_judge_log.csv"
    )

Processando estudo: NCT00236236
Retomando progresso: 1 estudos já processados.


Estudos:   0%|          | 0/1 [00:00<?, ?it/s]

Erro na API Groq: 'NoneType' object has no attribute 'replace'


Estudos: 100%|██████████| 1/1 [03:51<00:00, 231.12s/it]


Processando estudo: NCT01550107
Retomando progresso: 2 estudos já processados.


...

Processando estudo: NCT00176384
Retomando progresso: 8 estudos já processados.


Estudos: 100%|██████████| 1/1 [00:00<00:00, 94.32it/s]


Processando estudo: NCT01510652
Retomando progresso: 9 estudos já processados.


Estudos:   0%|          | 0/1 [00:00<?, ?it/s]

Estudos: 100%|██████████| 1/1 [04:31<00:00, 271.93s/it]


In [18]:
df_judge = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_judge.csv")
# df_judge = df_judge.head(0)
df_judge
# df_judge.to_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_judge.csv", index=False)

,nct_id,eligible_count,subject_ids
0,NCT03319472,16,"[10093609, 10194132, 10362783, 11079862, 11799..."
